# Profile likelihoods from `scan/`

Reads the driver's npz files in `scan/results/` (`profile_prod_ad_<POI><TAG>.npz` for 1D, `profile_prod_ad_2d_<a>_<b><TAG>[_partKofN].npz` for 2D) and plots them. Self-contained: numpy, scipy and matplotlib only.

- `chi2` is the profile (lower envelope over every start) on `poi_grid`; `sigma1` = (lo, best, hi) at 68%, `cl95` = (lo, hi) at 95%, `kind1` / `kind95` say two-sided, upper / lower limit or unconstrained.
- `xstar` holds the other parameters at each grid point in the driver's scaled coordinates (value = cen + sig * x); `physical()` below converts with the run's config.
- Neff always comes from LINX's own output per row (`derived_bbn[:, 1]`, `derived_neff`), never from a formula in Delta_Neff_init.

In [ ]:
import glob, os, importlib.util
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2 as chi2dist

RES = "/pscratch/sd/c/carag/ABCMB-k/scan/results"
LEV1 = {"68%": chi2dist.ppf(0.682689, 1), "95%": chi2dist.ppf(0.95, 1)}   # 1 parameter
LEV2 = {"68%": chi2dist.ppf(0.682689, 2), "95%": chi2dist.ppf(0.95, 2)}   # 2 parameters


def load_1d(tag):
    """{poi: npz} for every 1D profile of a run tag."""
    out = {}
    for f in sorted(glob.glob(f"{RES}/profile_prod_ad_*{tag}.npz")):
        poi = os.path.basename(f)[len("profile_prod_ad_"):-len(tag + ".npz")]
        if not poi.startswith("2d_"):
            out[poi] = np.load(f, allow_pickle=True)
    return out


def config_of(r):
    """the run's config dict (its file path is stored in the npz)."""
    spec = importlib.util.spec_from_file_location("cfg", str(r["config"]))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    return m.CONFIG


def physical(r, cfg, name):
    """a free parameter at each grid point, from the scaled xstar (BBN nuisances: cen 1 or 0, sig tau_n_sig or 1)."""
    j = list(r["nuis"]).index(name)
    cen = cfg["cen"].get(name, 1.0 if name == "tau_n_fac" else 0.0)
    sig = cfg["sig"].get(name, cfg["bbn_like"]["tau_n_sig"] if name == "tau_n_fac" else 1.0)
    return cen + sig * r["xstar"][..., j]


def x_axis(poi, r):
    """(values, label): Delta_Neff_init profiles are drawn against each row's LINX Neff, log10_m_chi against m_chi."""
    if poi == "Delta_Neff_init" and "derived_bbn" in r.files:
        return r["derived_bbn"][:, 1], "Neff (LINX, per row)"
    if poi == "log10_m_chi":
        return 10 ** r["poi_grid"], "m_chi [MeV]"
    return r["poi_grid"], poi


def summary(runs):
    for poi, r in runs.items():
        lo, best, hi = r["sigma1"]; lo95, hi95 = r["cl95"]
        kind = str(r["kind1"]) if "kind1" in r.files else "two-sided"
        print(f"{poi:16s} best {best:.5g}  68% [{lo:.5g}, {hi:.5g}]  95% [{lo95:.5g}, {hi95:.5g}]  ({kind})  "
              f"min chi2 {np.nanmin(r['chi2']):.2f}  converged {int(r['converged'].sum())}/{len(r['chi2'])}")
        if "sigma1_neff" in r.files:
            a, b, c = r["sigma1_neff"]; d, e = r["cl95_neff"]
            print(f"{'  as Neff':16s} best {b:.4f}  68% [{a:.4f}, {c:.4f}]  95% [{d:.4f}, {e:.4f}]  (LINX per row)")

## 1D profiles of one run

In [ ]:
TAG = "_v6bbn"
runs = load_1d(TAG)
summary(runs)

fig, axes = plt.subplots(2, 4, figsize=(17, 7.5)); axes = axes.ravel()
for ax, (poi, r) in zip(axes, runs.items()):
    x, lab = x_axis(poi, r)
    ax.plot(x, r["chi2"] - np.nanmin(r["chi2"]), "o-", ms=3)
    bad = ~r["converged"]
    ax.plot(x[bad], (r["chi2"] - np.nanmin(r["chi2"]))[bad], "rx", ms=8, label="not converged" if bad.any() else None)
    for name, v in LEV1.items():
        ax.axhline(v, ls=":", c="gray")
    if poi == "log10_m_chi":
        ax.set_xscale("log")
    ax.set_xlabel(lab); ax.set_ylabel(r"$\Delta\chi^2$"); ax.set_ylim(-0.2, 8); ax.set_title(f"{poi} {TAG}")
for ax in axes[len(runs):]:
    ax.axis("off")
fig.tight_layout()

## Neff across runs

`_v3` profiles Neff itself; the LINX runs profile Delta_Neff_init and are drawn at each row's LINX Neff. Each run is offset by its own minimum (different likelihoods).

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
for tag in ["_v3", "_v4linx", "_v6bbn"]:
    r1 = load_1d(tag)
    r = r1.get("Neff", r1.get("Delta_Neff_init"))
    if r is None:
        continue
    x = r["poi_grid"] if "Neff" in r1 else r["derived_bbn"][:, 1]
    ax.plot(x, r["chi2"] - np.nanmin(r["chi2"]), "o-", ms=3, label=tag.lstrip("_"))
for v in LEV1.values():
    ax.axhline(v, ls=":", c="gray")
ax.set_xlabel("Neff"); ax.set_ylabel(r"$\Delta\chi^2$"); ax.set_ylim(-0.2, 8); ax.legend()

## What the other parameters do along a profile

Pick a POI and any of its free parameters (`r['nuis']`), plus the LINX outputs (YHe, Neff, T_nu, Y_p(BBN), D/H) where the run has them. Also the convergence certificate per grid point.

In [ ]:
POI, SHOW = "Delta_Neff_init", ["omega_b", "h", "q1", "q2"]
r = runs[POI]; cfg = config_of(r); x, lab = x_axis(POI, r)
print("free parameters:", list(r["nuis"]))
n = len(SHOW) + (2 if "derived_bbn" in r.files else 0) + 1
fig, axes = plt.subplots(1, n, figsize=(3.3 * n, 3.2))
for ax, name in zip(axes, SHOW):
    ax.plot(x, physical(r, cfg, name), "o-", ms=3); ax.set_ylabel(name)
if "derived_bbn" in r.files:
    axes[len(SHOW)].plot(x, r["derived_bbn"][:, 3], "o-", ms=3); axes[len(SHOW)].set_ylabel("Y_p (BBN)")
    axes[len(SHOW) + 1].plot(x, r["derived_bbn"][:, 4], "o-", ms=3); axes[len(SHOW) + 1].set_ylabel("1e5 D/H")
axes[-1].semilogy(x, r["gnorm"], "o-", ms=3); axes[-1].axhline(float(r["gtol"]), c="r", ls=":")
axes[-1].set_ylabel("max |grad| (AD, scaled)")
for ax in axes:
    ax.set_xlabel(lab)
fig.tight_layout()

## EM-coupled WIMP (`lcdm_mchi_dneff_linx_bbnlike_plikfull.py`)

1D profiles of LCDM + Delta_Neff_init + log10_m_chi (tag `_v1emwimp`), and the 2D profile of (m_chi, Delta_Neff_init) (tag `_v1emwimp2d`; one part file per node). m_chi is open above (LCDM beyond ~30 MeV), so its interval is a lower limit. The 2D contours use the 2-parameter thresholds (2.30, 5.99).

In [ ]:
em = load_1d("_v1emwimp")
if not em:
    print("no _v1emwimp results yet")
else:
    summary(em)
    if "log10_m_chi" in em:
        r = em["log10_m_chi"]
        print(f"m_chi > {10 ** r['cl95'][0]:.2f} MeV (95%), > {10 ** r['sigma1'][0]:.2f} MeV (68%)")
        fig, ax = plt.subplots(figsize=(6.5, 4.5))
        ax.semilogx(10 ** r["poi_grid"], r["chi2"] - np.nanmin(r["chi2"]), "o-", ms=3)
        for v in LEV1.values():
            ax.axhline(v, ls=":", c="gray")
        ax.axvline(10 ** r["cl95"][0], c="orange", label="95% lower limit")
        ax.set_xlabel("m_chi [MeV]"); ax.set_ylabel(r"$\Delta\chi^2$"); ax.legend()

In [ ]:
def load_2d(pa, pb, tag):
    """(grid_a, grid_b, chi2, converged, Neff) on the full grid from the part files."""
    parts = sorted(glob.glob(f"{RES}/profile_prod_ad_2d_{pa}_{pb}{tag}*.npz"))
    if not parts:
        return None
    p0 = np.load(parts[0], allow_pickle=True); ga, gb = p0["grid_a"], p0["grid_b"]
    chi2 = np.full((len(ga), len(gb)), np.nan); conv = np.zeros(chi2.shape, bool); neff = np.full(chi2.shape, np.nan)
    for f in parts:
        r = np.load(f, allow_pickle=True); ja = r["ja"]
        chi2[ja], conv[ja] = r["chi2"], r["converged"]
        if "derived_bbn" in r.files:
            neff[ja] = r["derived_bbn"][..., 1]
    print(f"{len(parts)} part(s); grid points filled {np.isfinite(chi2).sum()}/{chi2.size}; converged {conv.sum()}")
    return ga, gb, chi2, conv, neff


g2 = load_2d("log10_m_chi", "Delta_Neff_init", "_v1emwimp2d")
if g2 is None:
    print("no _v1emwimp2d results yet")
else:
    ga, gb, chi2, conv, neff = g2
    d = chi2 - np.nanmin(chi2)
    M, DN = np.meshgrid(10 ** ga, gb, indexing="ij")
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    ax = axes[0]
    pc = ax.pcolormesh(M, DN, np.minimum(d, 20), shading="nearest", cmap="viridis_r")
    ax.contour(M, DN, d, levels=list(LEV2.values()), colors=["w", "w"], linestyles=["-", "--"])
    ax.plot(M[~conv], DN[~conv], "rx", ms=6)
    ax.set_xscale("log"); ax.set_xlabel("m_chi [MeV]"); ax.set_ylabel("Delta_Neff_init")
    ax.set_xticks([7, 10, 20, 30, 50]); ax.set_xticklabels(["7", "10", "20", "30", "50"])
    fig.colorbar(pc, ax=ax, label=r"$\Delta\chi^2$ (capped at 20)")
    ax = axes[1]                                   # the same rows placed at their LINX Neff
    ok = np.isfinite(d) & np.isfinite(neff)
    tc = ax.tricontourf(np.log10(M[ok]), neff[ok], np.minimum(d[ok], 20), levels=40, cmap="viridis_r")
    ax.tricontour(np.log10(M[ok]), neff[ok], d[ok], levels=list(LEV2.values()), colors=["w", "w"], linestyles=["-", "--"])
    ax.set_xlabel("log10 m_chi [MeV]"); ax.set_ylabel("Neff (LINX, per row)")
    fig.colorbar(tc, ax=ax, label=r"$\Delta\chi^2$")
    fig.tight_layout()
    # consistency: minimizing the 2D surface over Delta_Neff_init is a (coarser) 1D profile in m_chi
    if "log10_m_chi" in em:
        fig, ax = plt.subplots(figsize=(6.5, 4))
        r = em["log10_m_chi"]
        ax.semilogx(10 ** r["poi_grid"], r["chi2"] - np.nanmin(chi2), "o-", ms=3, label="1D profile")
        ax.semilogx(10 ** ga, np.nanmin(d, axis=1), "s", label="2D, min over Delta_Neff_init")
        ax.set_xlabel("m_chi [MeV]"); ax.set_ylabel(r"$\chi^2 - \min_{2D}$"); ax.legend()